# Trees and Tree Algorithms


<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/nsysu-math208/blob/master/static_files/presentations/09_Trees and Tree Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/phonchi/nsysu-math208/blob/master/static_files/presentations/09_Trees and Tree Algorithms.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

In [ ]:
from jupyterquiz import display_quiz

path = "questions/ch9/"

In [ ]:
# === DS09 animation loader. ===
# Each animation is rendered inside a self-contained iframe.  This avoids
# relying on the notebook frontend to execute <script> tags embedded directly
# in ordinary HTML output, which is blocked or unreliable in several frontends.
from functools import lru_cache
from html import escape
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display

_BASE = "https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/extra/animations09/"
_LOCAL_ANIM_DIRS = [
    Path("animations09"),
    Path("nsysu-math208/extra/animations09"),
    Path("../../extra/animations09"),
]

@lru_cache(maxsize=32)
def _fetch(name):
    for folder in _LOCAL_ANIM_DIRS:
        path = folder / name
        if path.exists():
            return path.read_text(encoding="utf-8")
    return urlopen(_BASE + name, timeout=15).read().decode("utf-8")

class _RawHTML:
    def __init__(self, html):
        self.html = html

    def _repr_html_(self):
        return self.html

def _anim_iframe(name, height=500):
    css = _fetch("ds09.css")
    js = _fetch("ds09.js").replace("</script>", "<\\/script>")
    panel = _fetch(name + ".html")
    srcdoc = f"""<!doctype html>
<html>
<head>
<meta charset=\"utf-8\">
<style>html,body{{margin:0;padding:0;background:transparent;}}</style>
<style>{css}</style>
</head>
<body class="rise-enabled">
{panel}
<script>{js}</script>
</body>
</html>"""
    return _RawHTML(
        f'<iframe title="DS09 {escape(name)} animation" '
        f'srcdoc="{escape(srcdoc, quote=True)}" '
        f'style="width:100%;height:{height}px;border:0;display:block;" '
        f'loading="lazy"></iframe>'
    )

def load_infra():
    # Kept for old cells.  The iframe loader now carries its own CSS/JS.
    return None

def load_anim(name):
    display(_anim_iframe(name))

_ = load_infra()


1. The Tree Abstract Data Type

2. Tree Traversals

3. Priority Queue and Binary Heap

4. Binary Search Trees

5. AVL tree


## 8.2 Introduction

<u>Trees</u> are used in many areas of computer science, including operating systems, graphics, database systems, and computer networking. A tree data structure has a root, branches, and leaves. The difference between a tree in nature and a tree in computer science is that a tree data structure has its root at the top and its leaves on the bottom!

Our first example of a tree is a classification tree from biology which shows an example of the biological classification of some animals.

This example demonstrates that trees are **hierarchical**. By hierarchical, we mean that trees are structured in layers with the more general things near the top and the more specific things near the bottom. The top of the hierarchy is the kingdom, the next layer of the tree (the "children" of the layer above) is the phylum, then the class, and so on. 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/biology.png" width="50%" /></center>

Notice that you can start at the top of the tree and follow a path made of circles and arrows all the way to the bottom. At each level of the tree we might ask ourselves a question and then follow the path that agrees with our answer. 

For example we might ask, "Is this animal a chordate or an arthropod?" If the answer is "chordate," then we follow that path and ask, "Is this chordate a mammal?" If not, we are stuck (but only in this simplified example). When we are at the mammal level we ask, "Is this mammal a primate or a carnivore?" We can keep following paths until we get to the very bottom of the tree where we have the common name!

A second property of trees is that all of the **children of one node are independent of the children of another node** while the third property is that the path to **each leaf node is unique**.

 We can specify a path from the root of the tree to a leaf that uniquely identifies each species in the animal kingdom, for example Animalia -> Chordata -> Mammalia -> Carnivora -> Felidae -> Felis -> catus.

Another example of a tree structure that you probably use every day is a file system. In a file system, directories, or folders, are structured as a tree. The file system tree enables you to follow a path from the root to any directory. That path will uniquely identify that subdirectory 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/directory.png" width="80%" /></center>

Note that we could take the entire subtree staring with `/etc/`, detach `etc/` from the root and reattach it under `usr/`. This would change the unique pathname to httpd from `/etc/httpd` to `/usr/etc/httpd`, but would not affect the contents or any children of the httpd directory!

## 8.3. Vocabulary and Definitions

- <u>Node</u>: A node is a fundamental part of a tree. It can have a name, which we call the **key**. A node may also have additional information. We call this additional information the value or payload. 

- <u>Edge</u>: An edge is another fundamental part of a tree. An edge connects two nodes to show that there is a relationship between them. Every node (except the root) is connected by **exactly one incoming edge from another node**. Each node may have several outgoing edges.

- <u>Root</u>: The root of the tree is the only node in the tree that has no incoming edges.

- <u>Path</u>: A path is an ordered list of nodes that are connected by edges, for example, Mammalia -> Carnivora -> Felidae -> Felis -> catus is a path.

- <u>Children</u>: The set of nodes that have incoming edges from the same node are said to be the children of that node.
- <u>Parent</u>: A node is the parent of all the nodes it connects to with outgoing edges. 

- <u>Sibling</u>: Nodes in the tree that are children of the same parent are said to be siblings. The nodes `etc/` and `usr/` are siblings in the file system tree shown in Figure 2.
- <u>Subtree</u>: A subtree is a set of nodes and edges comprised of a parent and all the descendants of that parent.

- <u>Leaf Node</u>: A leaf node is a node that has no children. 
- <u>Level</u>: The level of a node $n$ is the number of edges on the path from the root node to $n$.
- <u>Height</u>: The height of a tree is equal to the maximum level of any node in the tree.

**Definition One**: A tree consists of a set of nodes and a set of edges that connect pairs of nodes. A tree has the following properties:

1. One node of the tree is designated as the root node.

2. Every node $n$, except the root node, is connected by an edge from exactly one other node $p$, where $p$ is the parent of $n$.

3. A unique path traverses from the root to each node.

4. If each node in the tree has a maximum of two children, we say that the tree is a <u>binary tree</u>.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/treedef1.png" width="40%" /></center>

The arrowheads on the edges indicate the direction of the connection.

**Definition Two**: A tree is either empty or consists of a root and zero or more subtrees, each of which is also a tree. The root of each subtree is connected to the root of the parent tree by an edge. 

Figure below illustrates this recursive definition of a tree. Using the recursive definition of a tree, we know that the tree below has at least four nodes (if it is not empty) since each of the triangles representing a subtree must have a root. It may have many more nodes than that, but we do not know unless we look deeper into the tree!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/TreeDefRecursive.png" width="30%" /></center>

In [ ]:
display_quiz(path+"tree.json", max_width=800)

<IPython.core.display.Javascript object>

## Tree ADT

We can use the following functions to create and manipulate a binary tree:

- `BinaryTree()`:  creates a new instance of a binary tree.

- `get_root_val()`: returns the value stored in the current node.

- `set_root_val(val)`: stores the value in parameter `val` in the current node.

- `get_left_child()`: returns the binary tree corresponding to the left child of the current node.
- `get_right_child()`: returns the binary tree corresponding to the right child of the current node.

- `insert_left(val)`: creates a new binary tree and installs it as the left child of the current node.

- `insert_right(val)`: creates a new binary tree and installs it as the right child of the current node.

The key decision in implementing a tree is choosing a good internal storage technique. We have two very interesting possibilities, and we will examine both before choosing one. We call them <u>list of lists</u> and <u>nodes and references</u>.

## List of Lists Representation

In a list of lists tree, we will store the value of the root node as the first element of the list. The second element of the list will itself be a list that represents the left subtree. The third element of the list will be another list that represents the right subtree. To illustrate this storage technique, let's look at an example. Figure below shows a simple tree and the corresponding list implementation:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/smalltree.png" width="40%" /></center>

```python
my_tree = [
        "a",  # root
        ["b",  # left subtree
            ["d", [], []],
            ["e", [], []]
        ],
        ["c",  # right subtree
            ["f", [], []],
            []
        ],
```

Notice that we can access subtrees of the list using standard list indexing. The root of the tree is `my_tree[0]`, the left subtree of the root is `my_tree[1]`, and the right subtree is `my_tree[2]`.

The structure of a list representing a subtree adheres to the structure defined for a tree; the structure itself is recursive! A subtree that has a root value and two empty lists is a leaf node. Another nice feature of the list of lists approach is that it generalizes to a tree that has many subtrees. In the case where the tree is more than a binary tree, another subtree is just another list!

In Python, a binary tree can be improvised out of **nested lists**: `["a", ["b", ["d", [], []], ["e", [], []]], ["c", ["f", [], []], []]]` — the root value followed by the left and right subtrees. Statically-typed `C++` has no such heterogeneous nested literal, so we go straight to the *nodes and references* representation, which is the natural one in C++ anyway.

Let's formalize this definition of the tree data structure by providing some functions that make it easy for us to use lists as trees. Note that we are not going to define a binary tree class. The functions we will write will just help us manipulate a standard list as though we are working with a tree!

*(For reference, the Python helper was: `make_binary_tree(root)` returning `[root, [], []]`.)*

*(`insert_left` popped the old child list and nested it under the new child — pure list surgery.)*

Notice that to insert a left child, we first obtain the (possibly empty) list that corresponds to the current left child. We then add the new left child, installing the old left child as the left child of the new one!

*(`insert_right` and the getter/setter helpers worked the same way on index 2.)*

The nodes-and-references implementation on the next slides provides exactly the same behavior with pointers instead of nested lists.

## 8.4. Nodes and References

Our second method to represent a tree uses nodes and references. In this case we will define a class that has attributes for the root value as well as the left and right subtrees.

 Using nodes and references, we might think of the tree as being structured like the one shown below:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/treerecs.png" width="50%" /></center>

Since this representation more closely follows the object-oriented programming paradigm, we will continue to use this representation for the remainder of the chapter.

We will start out with a simple class definition for the nodes and references approach as shown below. The important thing to remember about this representation is that the attributes `left_child` and `right_child` will become references to other instances of the `BinaryTree` class. For example, when we insert a new left child into the tree, we create another instance of `BinaryTree` and modify `self.left_child` in the root to reference the new tree.

```cpp
class BinaryTree {
    public:
        string key;
        BinaryTree* leftChild;
        BinaryTree* rightChild;

        BinaryTree(string rootObj) {
            key = rootObj;
            leftChild = NULL;
            rightChild = NULL;
        }
};
```

Just as you can store any object you like in a list, the root object of a tree can be a reference to any object. For our early examples, we will store the name of the node as the root value. Using nodes and references to represent the tree above, we would create six instances of the `BinaryTree` class.

Next let's look at the functions we need to build the tree beyond the root node. To add a left child to the tree, we will create a new binary tree object and set the `left_child` attribute of the root to refer to this new object. 

```cpp
void insertLeft(string newNode) {
    if (leftChild == NULL) {
        leftChild = new BinaryTree(newNode);
    } else {
        BinaryTree* newChild = new BinaryTree(newNode);
        newChild->leftChild = leftChild;
        leftChild = newChild;
    }
}
```

Note that we consider two cases for insertion. The first case is characterized by a node with no existing left child. When there is no left child, simply add a node to the tree. The second case is characterized by a node with an existing left child. In the second case, we insert a node and push the existing child down one level in the tree.

```cpp
void insertRight(string newNode) {
    if (rightChild == NULL) {
        rightChild = new BinaryTree(newNode);
    } else {
        BinaryTree* newChild = new BinaryTree(newNode);
        newChild->rightChild = rightChild;
        rightChild = newChild;
    }
}
```

To round out the definition for a simple binary tree data structure, we will write accessor methods for the left and right children and for the root values:

```cpp
string getRootVal() {
    return key;
}
void setRootVal(string newKey) {
    key = newKey;
}
BinaryTree* getLeftChild() {
    return leftChild;
}
BinaryTree* getRightChild() {
    return rightChild;
}
```

Now that we have all the pieces to create and manipulate a binary tree, let's use them to check on the structure a bit more. Let's make a simple tree with node "a" as the root, and add nodes "b" and "c" as children

The complete class is in `pythonds3/cppds/trees/binary_tree.cpp`. Here it is in action — note that an empty child prints as `0` (the NULL pointer):

In [1]:
#include <iostream>
#include "dscpp/binarytree.hpp"   // the class of this section
using namespace std;

int main() {
    BinaryTree aTree("a");
    cout << aTree.getRootVal() << endl;
    cout << aTree.getLeftChild() << endl;      // NULL prints as 0
    aTree.insertLeft("b");
    cout << aTree.getLeftChild()->getRootVal() << endl;
    aTree.insertRight("c");
    cout << aTree.getRightChild()->getRootVal() << endl;
    aTree.getRightChild()->setRootVal("hello");
    cout << aTree.getRightChild()->getRootVal() << endl;
    return 0;
}

a

0

b

c

hello



## 8.5. Parse Tree

<u>Parse trees</u> can be used to represent real-world constructions like sentences or mathematical expressions. Figure below shows the hierarchical structure of a simple sentence. Representing a sentence as a tree structure allows us to work with the individual parts of the sentence by using subtrees.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/nlParse.png" width="35%" /></center>

We can also represent a mathematical expression such as $((7 + 3) \cdot (5 - 2))$ as a parse tree, as shown below:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/meParse.png" width="35%" /></center>

We know that multiplication has a higher precedence than either addition or subtraction. Because of the parentheses, we know that before we can do the multiplication we must evaluate the parenthesized addition and subtraction expressions. The hierarchy of the tree helps us understand the order of evaluation for the whole expression!

Before we can evaluate the top-level multiplication, we must evaluate the addition and the subtraction in the subtrees. The addition, which is the left subtree, evaluates to 10. The subtraction, which is the right subtree, evaluates to 3.

Using the hierarchical structure of trees, we can simply replace an entire subtree with one node once we have evaluated the expressions in the children. Applying this replacement procedure gives us the simplified tree shown below:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/meSimple.png" width="15%" /></center>

In the rest of this section we are going to examine parse trees in more detail. In particular we will look at

- How to build a parse tree from a fully parenthesized mathematical expression.

- How to evaluate the expression stored in a parse tree.

- How to recover the original mathematical expression from a parse tree.

The first step in building a parse tree is to break up the expression string into a list of tokens. There are four different kinds of tokens to consider: **left parentheses, right parentheses, operators, and operands.**

We know that whenever we read a left parenthesis we are starting a new expression, and hence we should create a new tree to correspond to that expression. Conversely, whenever we read a right parenthesis, we have finished an expression. 

We also know that operands are going to be leaf nodes and children of their operators. Finally, we know that every operator is going to have both a left and a right child. Using the information from above we can define four rules as follows:

1. If the current token is a `(`, add a new node as the left child of the current node, and descend to the left child.

2. If the current token is in the list `["+", "-", "/", "*"]`, set the root value of the current node to the operator represented by the current token. Add a new node as the right child of the current node and descend to the right child.

3. If the current token is a number, set the root value of the current node to the number and return to the parent.

4. If the current token is a `)`, go to the parent of the current node.

Let's look at an example of the rules outlined above in action. We will use the expression $(3 + (4 * 5))$. We will parse this expression into the following list of character tokens: `["(", "3", "+", "(", "4", "*", "5", ")", ")"]`. Initially we will start out with a parse tree that consists of an empty root node.

Figure below illustrates the structure and contents of the parse tree as each new token is processed:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp1.png" width="5%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp2.png" width="8%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp3.png" width="8%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp4.png" width="8%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp5.png" width="8%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp6.png" width="8%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp7.png" width="10%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildExp8.png" width="10%" /></center>

Let's walk through the example step by step:

1. Create an empty tree.

2. Read `(` as the first token. By rule 1, create a new node as the left child of the root. Make the current node this new child.

3. Read 3 as the next token. By rule 3, set the root value of the current node to 3 and go back up the tree to the parent.

4. Read `+` as the next token. By rule 2, set the root value of the current node to `+` and add a new node as the right child. The new right child becomes the current node.

5. Read `(` as the next token. By rule 1, create a new node as the left child of the current node. The new left child becomes the current node.

6. Read 4 as the next token. By rule 3, set the value of the current node to 4. Make the parent of 4 the current node.

7. Read `*` as the next token. By rule 2, set the root value of the current node to `*` and create a new right child. The new right child becomes the current node.

8. Read 5 as the next token. By rule 3, set the root value of the current node to 5. Make the parent of 5 the current node.

9. Read `)` as the next token. By rule 4 we make the parent of `*` the current node.

10. Read `)` as the next token. By rule 4 we make the parent of `+` the current node. At this point there is no parent for `+`, so we are done.

From the example above, it is clear that we need to keep track of the current node as well as the parent of the current node. The tree interface provides us with a way to get children of a node, through the `get_left_child()` and `get_right_child()` methods, but how can we keep track of the parent?

A simple solution to keeping track of parents as we traverse the tree is to **use a stack**. Whenever we want to descend to a child of the current node, we first push the current node on the stack. When we want to return to the parent of the current node, we pop the parent off the stack!

```cpp
BinaryTree* buildParseTree(string fpExpr) {
    stack<BinaryTree*> pStack;
    BinaryTree* exprTree = new BinaryTree("");
    pStack.push(exprTree);
    BinaryTree* currentTree = exprTree;

    stringstream ss(fpExpr);
    string i;
    while (ss >> i) {
        if (i == "(") {
            currentTree->insertLeft("");
            pStack.push(currentTree);
            currentTree = currentTree->getLeftChild();
        }
        ...
```

```cpp
        ...
        else if (i == "+" || i == "-" || i == "*" || i == "/") {
            currentTree->setRootVal(i);
            currentTree->insertRight("");
            pStack.push(currentTree);
            currentTree = currentTree->getRightChild();
        } else if (i == ")") {
            currentTree = pStack.top();
            pStack.pop();
        } else {
            currentTree->setRootVal(i);   // a number
            currentTree = pStack.top();
            pStack.pop();
        }
    }
    return exprTree;
}
```

*(complete listing: `dscpp/binarytree.hpp`)*

The four rules for building a parse tree are coded as the first four clauses of the `if..elif` statements. In each case you can see that the code implements the rule, as described above, with a few calls to the `BinaryTree` or `Stack` methods. The only error checking we do in this function is in the `else` clause where a `ValueError` exception will be raised if **we get a token from the list that we do not recognize.**

In [2]:
#include <iostream>
#include "dscpp/binarytree.hpp"   // buildParseTree (md listing above)
using namespace std;

int main() {
    BinaryTree* pt = buildParseTree("( 3 + ( 4 * 5 ) )");
    inorder(pt);   // defined and explained in the next section
    cout << endl;
    return 0;
}

3 + 4 * 5 



Now that we have built a parse tree, what can we do with it? As a first example, we will write a function to evaluate the parse tree and return the numerical result. To write this function, we will make use of the **hierarchical nature** of the tree. 

Recall that we can replace the original tree with the simplified tree shown in above Figure. This suggests that we can write an algorithm that evaluates a parse tree by **recursively evaluating each subtree**.

A natural base case for recursive algorithms that operate on trees is to check for a leaf node. In a parse tree, the leaf nodes will always be operands. Since numerical objects like integers and floating points require no further interpretation, the evaluate function can simply return the value stored in the leaf node. 

The recursive step that moves the function toward the base case is to call evaluate on both the left and the right children of the current node. The recursive call effectively moves us down the tree, toward a leaf node.

To put the results of the two recursive calls together, we can simply apply the operator stored in the parent node to the results returned from evaluating both children. The code for a recursive evaluate function is shown below:

```cpp
double evaluate(BinaryTree* parseTree) {
    BinaryTree* leftChild = parseTree->getLeftChild();
    BinaryTree* rightChild = parseTree->getRightChild();

    if (leftChild != NULL && rightChild != NULL) {
        string op = parseTree->getRootVal();
        if (op == "+") return evaluate(leftChild) + evaluate(rightChild);
        if (op == "-") return evaluate(leftChild) - evaluate(rightChild);
        if (op == "*") return evaluate(leftChild) * evaluate(rightChild);
        return evaluate(leftChild) / evaluate(rightChild);
    } else {
        return stod(parseTree->getRootVal());   // a leaf: a number
    }
}
```

First, we obtain references to the left and the right children of the current node. If both the left and right children evaluate to `None`, then we know that the current node is really a leaf node. If the current node is not a leaf node, look up the operator in the current node and apply it to the results from recursively evaluating the left and right children.

To implement the arithmetic, we use a dictionary with the keys `+`, `-`, `*`, and `/`. **The values stored in the dictionary are functions from Python's operator module.** The operator module provides us with the function versions of many commonly used operators. When we look up an operator in the dictionary, the corresponding function object is retrieved. Since the retrieved object is a function, we can call it in the usual way: `function(param1, param2)`. So the lookup operators`["+"](2, 2)` is equivalent to `operator.add(2, 2)`.

Finally, we will trace the evaluate function on the parse tree we created before. When we first call `evaluate()`, we pass the root of the entire tree as the parameter `parse_tree()`. Then we obtain references to the left and right children to make sure they exist. The recursive call takes place on line 16. 

In [3]:
#include <iostream>
#include "dscpp/binarytree.hpp"   // evaluate (md listing above)
using namespace std;

int main() {
    BinaryTree* pt = buildParseTree("( 3 + ( 4 * 5 ) )");
    cout << evaluate(pt) << endl;
    return 0;
}

23



We begin by looking up the operator in the root of the tree, which is `+`. The `+` operator maps to the `operator.add` function call, which takes two parameters. As usual for a Python function call, the first thing Python does is to evaluate the parameters that are passed to the function. In this case both parameters are recursive function calls to our evaluate function. Using left-to-right evaluation, the first recursive call goes to the left. In the first recursive call the evaluate function is given the left subtree. We find that the node has no left or right children, so we are in a leaf node. When we are in a leaf node, we just return the value stored in the leaf node as the result of the evaluation. In this case we return the integer 3.

At this point we have one parameter evaluated for our top-level call to operator.add. But we are not done yet. Continuing the left-to-right evaluation of the parameters, we now make a recursive call to evaluate the right child of the root. We find that the node has both a left and a right child so we look up the operator stored in this node, `*`, and call this function using the left and right children as the parameters. At this point you can see that both recursive calls will be to leaf nodes, which will evaluate to the integers 4 and 5 respectively. With the two parameters evaluated, we return the result of `operator.mul(4, 5)`. At this point we have evaluated the operands for the top level `+` operator and all that is left to do is finish the call to `operator.add(3, 20)`. The result of the evaluation of the entire expression tree for 
 is 23!

## 8.6. Tree Traversals

Now it is time to look at some additional usage patterns for trees. These usage patterns can be divided into three commonly used patterns to **visit all the nodes in a tree.**

The difference between these patterns is the order in which each node is visited. We call this visitation of the nodes a <u>tree traversal</u>. The three traversals we will look at are called <u>preorder</u>, <u>inorder</u>, and <u>postorder</u>. Let's start out by defining these three traversals more carefully, then look at some examples where these patterns are useful.

- Preorder: In a preorder traversal, we visit the root node first, then recursively do a preorder traversal of the left subtree, followed by a recursive preorder traversal of the right subtree.

- Inorder: In an inorder traversal, we recursively do an inorder traversal on the left subtree, visit the root node, and finally do a recursive inorder traversal of the right subtree.

- Postorder: In a postorder traversal, we recursively do a postorder traversal of the left subtree and the right subtree followed by a visit to the root node.

First let's look at the preorder traversal using a book as an example tree. The book is the root of the tree, and each chapter is a child of the root. Each section within a chapter is a child of the chapter, each subsection is a child of its section, and so on.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/booktree.png" width="50%" /></center>

Suppose that you wanted to read this book from front to back. The preorder traversal gives you exactly that ordering. 

Starting at the root of the tree (the `Book` node) we will follow the preorder traversal instructions. We recursively call preorder on the left child, in this case `Chapter1`. We again recursively call preorder on the left child to get to `Section 1.1`. Since `Section 1.1` has no children, we do not make any additional recursive calls. When we are finished with `Section 1.1`, we move up the tree to `Chapter 1`. At this point we still need to visit the right subtree of `Chapter 1`, which is `Section 1.2`. As before we visit the left subtree, which brings us to `Section 1.2.1`, then we visit the node for `Section 1.2.2`. With `Section 1.2` finished, we return to `Chapter 1`. Then we return to the `Book` node and follow the same procedure for `Chapter 2`.

The code for writing tree traversals is surprisingly elegant, largely because the traversals are written recursively. The code below shows a version of the preorder traversal written as an external function that takes a binary tree as a parameter. The external function is particularly elegant because our base case is simply to check if the tree exists. If the tree parameter is `None`, then the function returns without taking any action.

```cpp
void preorder(BinaryTree* tree) {
    if (tree != NULL) {
        cout << tree->getRootVal() << " ";
        preorder(tree->getLeftChild());
        preorder(tree->getRightChild());
    }
}
```

Output for our parse tree: `+ 3 * 4 5` — the prefix form of the expression.

In [ ]:
load_anim("preorder")

We can also implement preorder as a method of the BinaryTree class. The code for implementing preorder as an internal method. Notice what happens when we move the code from external to internal. In general, we just replace `tree` with `self`. However, we also need to modify the base case. The internal method must check for the existence of the left and the right children before making the recursive call to preorder.

As a method of the `BinaryTree` class, the traversal calls itself on the child subtrees:

```cpp
void preorder() {
    cout << key << endl;
    if (leftChild != NULL) {
        leftChild->preorder();
    }
    if (rightChild != NULL) {
        rightChild->preorder();
    }
}
```

Implementing preorder as an external function is probably better in this case. The reason is that you very rarely want to just traverse the tree. In most cases you are going to want to accomplish something else while using one of the basic traversal patterns.

In fact, we will see in the next example that the postorder traversal pattern follows very closely with the code we wrote earlier to evaluate a parse tree. Therefore we will write the rest of the traversals as external functions.

```cpp
void postorder(BinaryTree* tree) {
    if (tree != NULL) {
        postorder(tree->getLeftChild());
        postorder(tree->getRightChild());
        cout << tree->getRootVal() << " ";
    }
}
```

Output: `3 4 5 * +` — the postfix form.

In [ ]:
load_anim("postorder")

We have already seen a common use for the postorder traversal, namely **evaluating a parse tree**. Assuming our binary tree is going to store only expression tree data, rewrite the evaluation function, but model it even more closely on the postorder code, we have:

```cpp
double postordereval(BinaryTree* tree) {
    if (tree == NULL) return 0;
    if (tree->getLeftChild() != NULL && tree->getRightChild() != NULL) {
        double result1 = postordereval(tree->getLeftChild());
        double result2 = postordereval(tree->getRightChild());
        string op = tree->getRootVal();
        if (op == "+") return result1 + result2;
        if (op == "-") return result1 - result2;
        if (op == "*") return result1 * result2;
        return result1 / result2;
    }
    return stod(tree->getRootVal());
}
```

Output: `23` — evaluation is just a post-order traversal.

Note that except that instead of printing the key at the end of the function, we return it. This allows us to save the values returned from the recursive calls. We then use these saved values along with the operator.

The final traversal we will look at in this section is the inorder traversal. In the inorder traversal we visit the left subtree, followed by the root, and finally the right subtree.

```cpp
void inorder(BinaryTree* tree) {
    if (tree != NULL) {
        inorder(tree->getLeftChild());
        cout << tree->getRootVal() << " ";
        inorder(tree->getRightChild());
    }
}
```

Output: `3 + 4 * 5` — the familiar infix form (without parentheses).

In [ ]:
load_anim("inorder")

If we perform a simple inorder traversal of a parse tree, we get our original expression back without any parentheses. Let's modify the basic inorder algorithm to allow us to recover the fully parenthesized version of the expression. The only modifications we will make to the basic template are as follows.

Print a left parenthesis before the recursive call to the left subtree, and print a right parenthesis after the recursive call to the right subtree:

```cpp
string printExp(BinaryTree* tree) {
    string result = "";
    if (tree != NULL) {
        result = "(" + printExp(tree->getLeftChild());
        result = result + tree->getRootVal();
        result = result + printExp(tree->getRightChild()) + ")";
    }
    return result;
}
```

In [4]:
#include <iostream>
#include "dscpp/binarytree.hpp"   // printExp (md listing above)
using namespace std;

int main() {
    BinaryTree* pt = buildParseTree("( 3 + ( 4 * 5 ) )");
    cout << printExp(pt) << endl;
    return 0;
}

((3)+((4)*(5)))



In [ ]:
display_quiz(path+"traversal.json", max_width=800)

<IPython.core.display.Javascript object>

#### Exercise 1: Clean up the `print_exp()` function so that it does not include an extra set of parentheses around each number.

In [5]:
#include <iostream>
#include "dscpp/binarytree.hpp"
using namespace std;

// Modify printExp so leaf numbers are not wrapped in parentheses
string printExp2(BinaryTree* tree) {
    string result = "";
    if (tree != NULL) {
        result = "(" + printExp2(tree->getLeftChild());
        result = result + tree->getRootVal();
        result = result + printExp2(tree->getRightChild()) + ")";
    }
    return result;
}

int main() {
    BinaryTree* pt = buildParseTree("( 3 + ( 4 * 5 ) )");
    cout << printExp2(pt) << endl;
    return 0;
}

((3)+((4)*(5)))



Goal: print `(3 + (4 * 5))` instead of `((3) + ((4) * (5)))`.

## 8.7. Priority Queues with Binary Heaps

You can probably think of a couple of easy ways to implement a <U>priority queue</U> using sorting functions and lists. However, inserting into a list is $O(n)$ and sorting a list is $O(n \log{n})$. 

We can do better. The classic way to implement a priority queue is using a data structure called a <u>binary heap</u>. A binary heap will allow us both to enqueue and dequeue items in $O(\log{n})$.

The binary heap is interesting to study because when we diagram the heap it looks a lot like a tree, but when we implement it we use only a single list as an internal representation. The binary heap has two common variations: the <u>min heap</u>, in which the smallest key value is always at the front, and the <u>max heap</u>, in which the largest key value is always at the front. In this section, we will implement the min heap.

## 8.9. Binary Heap Operations

- `BinaryHeap()`: creates a new empty binary heap.

- `insert(k)`: adds a new item to the heap.

- `get_min()`: returns the item with the minimum key value, leaving the item in the heap.

- `delete()`: returns the item with the minimum key value, removing the item from the heap.

- `is_empty()`: returns `True` if the heap is empty, `False` otherwise.

- `size()`: returns the number of items in the heap.

- `heapify(list)`: builds a new heap from a list of keys.

In [6]:
#include <iostream>
#include "dscpp/binaryheap.hpp"   // the class of this section
using namespace std;

int main() {
    BinaryHeap myHeap;
    myHeap.insert(5);
    myHeap.insert(7);
    myHeap.insert(3);
    myHeap.insert(11);

    cout << myHeap.delet() << endl;
    cout << myHeap.delet() << endl;
    cout << myHeap.delet() << endl;
    cout << myHeap.delet() << endl;
    return 0;
}

3

5

7

11



In [7]:
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

class BinaryHeap {
    public:
        vector<int> heap;

        bool isEmpty() { return heap.empty(); }
        void percUp(int i) {
            while ((i - 1) / 2 >= 0 && i > 0) {
                int parentIdx = (i - 1) / 2;
                if (heap[i] < heap[parentIdx]) {
                    swap(heap[i], heap[parentIdx]);
                } else {
                    break;
                }
                i = parentIdx;
            }
        }
        void insert(int item) {
            heap.push_back(item);
            percUp(heap.size() - 1);
        }
        int getMinChild(int i) {
            if (2 * i + 2 > (int)heap.size() - 1) {
                return 2 * i + 1;
            }
            if (heap[2 * i + 1] < heap[2 * i + 2]) {
                return 2 * i + 1;
            }
            return 2 * i + 2;
        }
        void percDown(int i) {
            while (2 * i + 1 < (int)heap.size()) {
                int smChild = getMinChild(i);
                if (heap[i] > heap[smChild]) {
                    swap(heap[i], heap[smChild]);
                } else {
                    break;
                }
                i = smChild;
            }
        }
        int delet() {
            swap(heap[0], heap[heap.size() - 1]);
            int result = heap.back();
            heap.pop_back();
            percDown(0);
            return result;
        }
        void heapify(vector<int> notAHeap) {
            heap = notAHeap;
            int i = heap.size() / 2 - 1;
            while (i >= 0) {
                percDown(i);
                i = i - 1;
            }
        }
        void print() {
            for (int x : heap) cout << x << " ";
            cout << endl;
        }
};

int main() {
    BinaryHeap myHeap;
    int items[] = {10, 4, 9, 8, 12, 15, 3, 5, 14, 18};
    for (int item : items) {
        myHeap.insert(item);
        myHeap.print();
    }
    return 0;
}

10 

4 10 

4 10 9 

4 8 9 10 

4 8 9 10 12 

4 8 9 10 12 15 

3 8 4 10 12 15 9 

3 5 4 8 12 15 9 10 

3 5 4 8 12 15 9 10 14 

3 5 4 8 12 15 9 10 14 18 



Notice that no matter what order we add items to the heap, the smallest is removed each time!

## 8.10. Binary Heap Implementation

In order to make our heap work efficiently, we will take advantage of the logarithmic nature of the binary tree to represent our heap.

In order to guarantee logarithmic performance, we must keep our tree **balanced**. A balanced binary tree has roughly the same number of nodes in the left and right subtrees of the root. In our heap implementation we keep the tree balanced by creating a <u>complete binary tree</u>.

A complete binary tree is a tree in which each level has all of its nodes. The exception to this is the bottom level of the tree, which we fill in from left to right. Figure below shows an example of a complete binary tree.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/compTree.png" width="40%" /></center>

Another interesting property of a complete tree is that we can represent it using a **single list**. We do not need to use nodes and references or even lists of lists. Because the tree is complete, the left child of a parent (at position $p$) is the node that is found in position $2p+1$ in the list!

Similarly, the right child of the parent is at position $2p+2$ in the list. To find the parent of any node in the tree, we can simply use integer division. Given that a node is at position $n$ in the list, the parent is at position $(n - 1) // 2$.

Figure below shows a complete binary tree and also gives the list representation of the tree. Note the $2p+1$ and $2p+2$ relationship between parent and children. 

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/heapOrder.png" width="40%" /></center>

The list representation of the tree, along with the full structure property, allows us to efficiently traverse a complete binary tree using only a few simple mathematical operations. 

The method that we will use to store items in a heap relies on maintaining the <u>heap order property</u>. The heap order property is as follows: in a heap, for every node $x$ with parent $p$, the key in $p$ is smaller than or equal to the key in $x$. Figure above also illustrates a complete binary tree that has the heap order property.

We now begin our implementation of a binary heap with the constructor. Since the entire binary heap can be represented by a single list, all the constructor will do is initialize the list:

```cpp
class BinaryHeap {
    public:
        vector<int> heap;   // the complete tree, stored flat

        BinaryHeap() {}
        bool isEmpty() {
            return heap.empty();
        }
};
```

With the root at index 0, node `i` has children at `2i + 1` and `2i + 2`, and its parent at `(i - 1) / 2`.

The next method we will implement is `insert()`. The easiest, and most efficient, way to add an item to a list is to simply append the item to the end of the list. The good news about appending is that it guarantees that we will maintain the <u>complete tree property</u>. The bad news about appending is that we will very likely **violate the heap structure property.**

However, it is possible to write a method that will allow us to regain the heap structure property by comparing the newly added item with its parent. If the newly added item is less than its parent, then we can swap the item with its parent.

Figure below shows the series of swaps needed to percolate the newly added item up to its proper position in the tree.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percUp1.png" width="40%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percUp2.png" width="40%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percUp3.png" width="40%" /></center>

Notice that when we percolate an item up, we are restoring the heap property between the newly added item and the parent. We are also preserving the heap property for any siblings. Of course, if the newly added item is very small, we may still need to swap it up another level. In fact, we may need to keep swapping until we get to the top of the tree!

```cpp
void percUp(int i) {
    while (i > 0) {
        int parentIdx = (i - 1) / 2;
        if (heap[i] < heap[parentIdx]) {
            swap(heap[i], heap[parentIdx]);
        } else {
            break;
        }
        i = parentIdx;
    }
}
```

We are now ready to write the `insert()` method. Most of the work in the `insert()` method is really done by `_perc_up()`. Once a new item is appended to the tree, `_perc_up()` takes over and positions the new item properly.

In [ ]:
load_anim("heap_insert")

```cpp
void insert(int item) {
    heap.push_back(item);
    percUp(heap.size() - 1);
}
```

With the `insert()` method properly defined, we can now look at the `delete()` method. Since the heap property requires that the root of the tree be the smallest item in the tree, finding the minimum item is easy.

The hard part of delete is restoring full compliance with the heap structure and heap order properties after the root has been removed. We can restore our heap in two steps.

First, we will restore the root item by taking the last item in the list and moving it to the root position. Moving the last item maintains our heap structure property. However, we have probably destroyed the heap order property of our binary heap. 

Second, we will restore the heap order property by pushing the new root node down the tree to its proper position. Figure below shows the series of swaps needed to move the new root node to its proper position in the heap.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percDown1.png" width="38%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percDown2.png" width="38%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percDown3.png" width="38%" /></center>

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/percDown4.png" width="38%" /></center>

In order to maintain the heap order property, all we need to do is swap the root with its smaller child that is less than the root. After the initial swap, we may repeat the swapping process with a node and its children until the node is swapped into a position on the tree where it is already less than both children.

```cpp
void percDown(int i) {
    while (2 * i + 1 < (int)heap.size()) {
        int smChild = getMinChild(i);
        if (heap[i] > heap[smChild]) {
            swap(heap[i], heap[smChild]);
        } else {
            break;
        }
        i = smChild;
    }
}

int getMinChild(int i) {
    if (2 * i + 2 > (int)heap.size() - 1) {
        return 2 * i + 1;
    }
    if (heap[2 * i + 1] < heap[2 * i + 2]) {
        return 2 * i + 1;
    }
    return 2 * i + 2;
}
```

The code for the delete operation is in below. Note that once again the hard work is handled by a helper function, in this case `_perc_down()`.

```cpp
int delet() {   // C++ reserves the word delete!
    swap(heap[0], heap[heap.size() - 1]);
    int result = heap.back();
    heap.pop_back();
    percDown(0);
    return result;
}
```

In [ ]:
load_anim("heap_extract")

To finish our discussion of binary heaps, we will look at a method to build an entire heap from a list of keys. The first method you might think of may be like the following. Given a list of keys, you could easily build a heap by inserting each key one at a time. Since you are starting with an empy list, it is sorted and you could use binary search to find the right position to insert the next key at a cost of approximately $O(\log{n})$ operations. 

However, remember that inserting an item in the middle of the list may require $O(n)$ operations to shift the rest of the list over to make room for the new key.

Therefore, to insert $n$ keys into the heap would require a total of $O(n^2)$ operations. However, if we start with an entire list then we can build the whole heap in $O(n)$ operations.

```cpp
void heapify(vector<int> notAHeap) {
    heap = notAHeap;   // copy the vector
    int i = heap.size() / 2 - 1;   // last non-leaf node
    while (i >= 0) {
        percDown(i);
        i = i - 1;
    }
}
```

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/buildheap.png" width="45%" /></center>

Figure above shows the swaps that the `hepify()` method makes as it moves the nodes in an initial tree of `[9, 6, 5, 2, 3]` into their proper positions. Although we start out in the middle of the tree and work our way back toward the root, the `_perc_down()` method ensures that the largest child is always moved down the tree. Because the heap is a complete binary tree, any nodes past the halfway point will be leaves and therefore have no children!

Notice that when `i = 0`, we are percolating down from the root of the tree, so this may require multiple swaps. As you can see in the rightmost two trees of the figure, first the 9 is moved out of the root position, but after 9 is moved down one level in the tree, `_perc_down()` ensures that we check the next set of children farther down in the tree to ensure that it is pushed as low as it can go. 

In this case it results in a second swap with 3. Now that 9 has been moved to the lowest level of the tree, no further swapping can be done.

In [ ]:
load_anim("heap_build")

```
start  [9, 6, 5, 2, 3]
i = 1  [9, 2, 5, 6, 3]
i = 0  [2, 3, 5, 6, 9]
```

In [8]:
#include <iostream>
#include "dscpp/binaryheap.hpp"
using namespace std;

int main() {
    BinaryHeap aHeap;
    aHeap.heapify({10, 4, 9, 8, 12, 15, 3, 5, 14, 18});
    aHeap.print();
    return 0;
}

3 4 9 5 12 15 10 8 14 18 



The assertion that we can build the heap in $O(n)$ may seem a bit mysterious at first, and a proof is beyond the scope of this course. However, the key to understanding that you can build the heap in $O(n)$ is to remember that the $\log{n}$ factor is derived from the height of the tree. For most of the work in `heapify()`, the tree is shorter than $\log{n}$.

In [ ]:
display_quiz(path+"binary_heap.json", max_width=800)

<IPython.core.display.Javascript object>

Using the fact that you can build a heap from a list in $O(n)$ time, you will construct a sorting algorithm that uses a heap and sorts a list in $O(n\log{n})$ as an exercise at the end of this chapter.

https://visualgo.net/en/heap

#### Exercise 2: Using the `heapify()` and `delete()` method, write a sorting function that can sort a list in $O(n\log n)$ time.

In [9]:
#include <iostream>
#include <vector>
#include "dscpp/binaryheap.hpp"
using namespace std;

vector<int> heapSort(vector<int> unsortedList) {
    BinaryHeap heap;
    vector<int> sortedList;
    ____;                  // 1. build the heap in O(n)
    while (____) {         // 2. repeatedly delete the minimum, O(log n) each
        ____;
    }
    return sortedList;
}
int main() {
    for (int x : heapSort({10, 3, 5, 1, 15, 7, 9, 2, 8})) cout << x << " ";
    return 0;
}

/tmp/tmpa_zgg0k7.cpp: In function ‘std::vector<int> heapSort(std::vector<int>)’:
/tmp/tmpa_zgg0k7.cpp:11:5: error: ‘____’ was not declared in this scope
   11 |     ____;
      |     ^~~~



[C++ kernel] Error: Unable to compile the source code. Return error: 0x1.

Available `BinaryHeap` methods: `insert`, `delet`, `heapify`, `isEmpty`, `percUp`, `percDown`, `getMinChild`, `print`.

Expected output: `Sorted list: 1 2 3 5 7 8 9 10 15`

## 8.11. Binary Search Trees

We have already seen two different ways to get key-value pairs in a collection. Recall that these collections implement the **map** abstract data type. The two implementations of the map ADT that we have discussed were binary search on a list and hash tables. 

In this section we will study <u>binary search trees</u> as yet another way to map from a key to a value. In this case we are not interested in the exact placement of items in the tree, but we are interested in using the binary tree structure to provide for efficient searching.

## 8.12. Search Tree Operations

Before we look at the implementation, let's review the interface provided by the map ADT. You will notice that this interface is very similar to the dictionary.

- `Map()`: creates a new empty map.

- `put(key, val)`: adds a new key–value pair to the map. If the key is already in the map, it replaces the old value with the new value.

- `get(key)`: takes a key and returns the matching value stored in the map or `None` otherwise.

- `del`: deletes the key–value pair from the map using a statement of the form `del map[key]`.

- `size()`: returns the number of key–value pairs stored in the map.

- `in`: return `True` for a statement of the form key in map if the given key is in the map, `False` otherwise.

## 8.13. Search Tree Implementation

A binary search tree (BST) relies on the property that keys that are less than the parent are found in the left subtree, and keys that are greater than the parent are found in the right subtree. We will call this the <u>BST property</u>.

Figure below illustrates this property of a binary search tree, showing the keys without any associated values. 

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/simpleBST.png" width="30%"/>
</center>

Notice that the property holds for each parent and child. All of the keys in the left subtree are less than the key in the root. All of the keys in the right subtree are greater than the root.

Now that you know what a binary search tree is, we will look at how a binary search tree is constructed. The search tree above represents the nodes that exist after we have inserted the following keys in the order shown: $70, 31, 93, 94, 14, 23, 73$. Since 70 was the first key inserted into the tree, it is the root.

Next, 31 is less than 70, so it becomes the left child of 70. Next, 93 is greater than 70, so it becomes the right child of 70. Now we have two levels of the tree filled, so the next key is going to be the left or right child of either 31 or 93.

Since 94 is greater than 70 and 93, it becomes the right child of 93. Similarly 14 is less than 70 and 31, so it becomes the left child of 31. 23 is also less than 31, so it must be in the left subtree of 31. However, it is greater than 14, so it becomes the right child of 14.

In [ ]:
load_anim("bst_search")

To implement the binary search tree, we will use the **nodes and references** approach similar to the one we used to implement the linked list and the expression tree. 

However, because we must be able create and work with a binary search tree that is empty, our implementation will use two classes. The first class we will call `BinarySearchTree`, and the second class we will call `TreeNode`.

The `BinarySearchTree` class has a reference to the `TreeNode` that is the root of the binary search tree. In most cases the external methods defined in the outer class simply check to see if the tree is empty. If there are nodes in the tree, the request is just passed on to a private method defined in the `BinarySearchTree` class that takes the root as a parameter.

In the case where the tree is empty or we want to delete the key at the root of the tree, we must take special action.

```cpp
class BinarySearchTree {
    public:
        TreeNode* root;
        int size;

        BinarySearchTree() {
            root = NULL;
            size = 0;
        }
        int length() {
            return size;
        }
};
```

The `TreeNode` class provides many helper methods that make the work done in the `BinarySearchTree` class methods much easier. 

Many of these helper methods help to classify a node according to its own position as a child (left or right) and the kind of children the node has. 

```cpp
class TreeNode {
    public:
        string key;
        string value;
        TreeNode* leftChild;
        TreeNode* rightChild;
        TreeNode* parent;

        TreeNode(string k, string v, TreeNode* p = NULL) {
            key = k;
            value = v;
            leftChild = NULL;
            rightChild = NULL;
            parent = p;
        }
    ...
```

```cpp
    ...
        bool isLeftChild() {
            return parent != NULL && parent->leftChild == this;
        }
        bool isRightChild() {
            return parent != NULL && parent->rightChild == this;
        }
        bool isRoot() {
            return parent == NULL;
        }
        bool isLeaf() {
            return leftChild == NULL && rightChild == NULL;
        }
        bool hasAnyChild() {
            return leftChild != NULL || rightChild != NULL;
        }
};
```

The `TreeNode` class will also explicitly keep track of the parent as an attribute of each node. You will see why this is important when we discuss the implementation for the `del` operator.

Another interesting aspect is that we use optional parameters which make it easy for us to create a `TreeNode` under several different circumstances. 

Sometimes we will want to construct a new `TreeNode` that already has both a parent and a child (e.g. left) and in that case we can pass parent and left_child as parameters. 

At other times we will just create a TreeNode with the key-value pair, and we will not pass any parameters for parent or child. In this case, the default values of the optional parameters are used.

Now that we have the `BinarySearchTree` shell and the `TreeNode`, it is time to write the `put()` method that will allow us to build our binary search tree. The method is a method of the `BinarySearchTree` class. This method will check to see if the tree already has a root. If there is not a root, then `put()` will create a new `TreeNode` and install it as the root of the tree.

If a root node is already in place, then `put()` calls the private recursive helper method `_put()` to search the tree according to the following algorithm.

- Starting at the root of the tree, search the binary tree comparing the new key to the key in the current node. If the new key is less than the current node, search the left subtree. If the new key is greater than the current node, search the right subtree.

- When there is no left or right child to search, we have found the position in the tree where the new node should be installed.

- To add a node to the tree, create a new `TreeNode` object and insert the object at the point discovered in the previous step.

```cpp
void put(string key, string value) {
    if (root != NULL) {
        _put(key, value, root);
    } else {
        root = new TreeNode(key, value);
    }
    size = size + 1;
}

void _put(string key, string value, TreeNode* currentNode) {
    if (key < currentNode->key) {
        if (currentNode->leftChild != NULL) {
            _put(key, value, currentNode->leftChild);
        } else {
            currentNode->leftChild = new TreeNode(key, value, currentNode);
        }
    } else {
        if (currentNode->rightChild != NULL) {
            _put(key, value, currentNode->rightChild);
        } else {
            currentNode->rightChild = new TreeNode(key, value, currentNode);
        }
    }
}
```

Notice that when a new child is inserted into the tree, the `current_node` is passed to the new tree as the parent.

One important problem with our implementation of insertion is that duplicate keys are not handled properly. A duplicate key will create a new node with the same key value in the right subtree of the node having the original key. The result of this is that the node with the new key will never be found during a search!

A better way to handle the insertion of a duplicate key is **for the value associated with the new key to replace the old value.** We leave fixing this bug as an exercise for you.

With the `put()` method defined, we can easily overload the `[]` operator for assignment. This allows us to write statements like `my_zip_tree['Plymouth'] = 55446`, just like a Python dictionary!

In `C++` we can overload `operator[]` so that `myTree["a"] = "apple";` calls `put()` — the same sugar Python provides with `__setitem__`.

Figure below illustrates the process for inserting a new node into a binary search tree. The lightly shaded nodes indicate the nodes that were visited during the insertion process.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bstput.png" width="45%"/>
</center>

In [ ]:
load_anim("bst_insert")

The `get()` method is even easier than the `put()` method because it simply searches the tree recursively until it gets to a non-matching leaf node or finds a matching key. When a matching key is found, the value stored in the payload of the node is returned.

```cpp
string get(string key) {
    if (root != NULL) {
        TreeNode* result = _get(key, root);
        if (result != NULL) {
            return result->value;
        }
    }
    return "";
}

TreeNode* _get(string key, TreeNode* currentNode) {
    if (currentNode == NULL) {
        return NULL;
    }
    if (currentNode->key == key) {
        return currentNode;
    } else if (key < currentNode->key) {
        return _get(key, currentNode->leftChild);
    } else {
        return _get(key, currentNode->rightChild);
    }
}
```

**Notice that the `_get()` method returns a `TreeNode` to `get()`**, this allows `_get()` to be used as a flexible helper method for other `BinarySearchTree` methods that may need to make use of other data from the `TreeNode` besides the payload!

We can implement two to related methods as follows:

```cpp
bool contains(string key) {
    return _get(key, root) != NULL;
}
```

Finally, we turn our attention to the most challenging operation on the binary search tree, the deletion of a key. The first task is to find the node to delete by searching the tree. If the tree has more than one node we search using the `_get()` method to find the `TreeNode` that needs to be removed. 

If the tree only has a single node, that means we are removing the root of the tree, but we still must check to make sure the key of the root matches the key that is to be deleted!

```cpp
void remove(string key) {   // C++ reserves the word delete
    if (size > 1) {
        TreeNode* nodeToRemove = _get(key, root);
        if (nodeToRemove != NULL) {
            _delete(nodeToRemove);
            size = size - 1;
        } else {
            throw invalid_argument("Error, key not in tree");
        }
    } else if (size == 1 && root->key == key) {
        root = NULL;
        size = size - 1;
    } else {
        throw invalid_argument("Error, key not in tree");
    }
}
```

Once we've found the node containing the key we want to delete, there are three cases that we must consider:

1. The node to be deleted has no children

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bstdel1.png" width="45%"/>
</center>

The first case is straightforward. If the current node has no children, all we need to do is delete the node and remove the reference to this node in the parent.

```cpp
if (currentNode->isLeaf()) {   // removing a leaf
    if (currentNode == currentNode->parent->leftChild) {
        currentNode->parent->leftChild = NULL;
    } else {
        currentNode->parent->rightChild = NULL;
    }
    delete currentNode;
}
```

2. The node to be deleted has only one child

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bstdel2.png" width="40%"/>
</center>

The second case is only slightly more complicated. If a node has only a single child, then we can simply promote the child to take the place of its parent. Since the cases are symmetric with respect to either having a left or right child,  we will just discuss the case where the current node has a left child.

1. If the current node is a left child, then we only need to update the parent reference of the left child to point to the parent of the current node, and then update the left child reference of the parent to point to the current node's left child.

2. If the current node is a right child, then we only need to update the parent reference of the left child to point to the parent of the current node, and then update the right child reference of the parent to point to the current node’s left child.

3. If the current node has no parent, it must be the root. In this case we will just replace the `key`, `value`, `left_child`, and `right_child` data by calling the `replace_value()` method on the root.

```cpp
else {   // removing a node with one child
    if (currentNode->leftChild != NULL) {
        if (currentNode->isLeftChild()) {
            currentNode->leftChild->parent = currentNode->parent;
            currentNode->parent->leftChild = currentNode->leftChild;
        } else if (currentNode->isRightChild()) {
            currentNode->leftChild->parent = currentNode->parent;
            currentNode->parent->rightChild = currentNode->leftChild;
        }
    }
    // (mirror image for a right child)
    delete currentNode;
}
```

3. The node to be deleted has two children

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bstdel3.png" width="40%"/>
</center>

The third case is the most difficult case to handle. If a node has two children, then it is unlikely that we can simply promote one of them to take the node's place!

We can, however, search the tree for a node that can be used to replace the one scheduled for deletion. What we need is a node that will preserve the BST relationships for both of the existing left and right subtrees. The node that will do this is the node that has the next-largest key in the tree. We call this node the <u>successor</u>.

The successor is guaranteed to have no more than one child, so we know how to remove it using the two cases for deletion that we have already implemented. Once the successor has been removed, we simply put it in the tree in place of the node to be deleted. We make use of the helper methods `find_successor()` and `splice_out()` to find and remove the successor. The reason we use `splice_out()` is that it goes directly to the node we want to splice out and makes the right changes. 

```cpp
else if (currentNode->hasBothChildren()) {   // removing a node with two children
    TreeNode* successor = currentNode->findSuccessor();
    successor->spliceOut();
    currentNode->key = successor->key;
    currentNode->value = successor->value;
    delete successor;
}
```

```cpp
TreeNode* findSuccessor() {   // a method of the TreeNode class
    TreeNode* successor = NULL;
    if (rightChild != NULL) {
        successor = rightChild->findMin();
    } else if (parent != NULL) {
        if (isLeftChild()) {
            successor = parent;
        } else {
            parent->rightChild = NULL;
            successor = parent->findSuccessor();
            parent->rightChild = this;
        }
    }
    return successor;
}
```

For deletion we only ever need the simple case — the node has a right subtree, so the successor is the minimum of that subtree:

```cpp
TreeNode* findSuccessor() {
    return rightChild->findMin();
}
```

This code makes use of the same properties of binary search trees that cause an **inorder traversal** to print out the nodes in the tree from smallest to largest. In general, there are three cases to consider when looking for the successor:

1. If the node has a right child, then the successor is the smallest key in the right subtree.

2. If the node has no right child and is the left child of its parent, then the parent is the successor.

3. If the node is the right child of its parent, and itself has no right child, then the successor to this node is the successor of its parent, excluding this node.

But here we only care about the first one since we have children!

The first condition is the only one that matters for us when deleting a node from a binary search tree. However, the find_successor method has other uses that we will explore in the exercises at the end of this chapter.

```cpp
TreeNode* findMin() {   // a method of the TreeNode class
    TreeNode* current = this;
    while (current->leftChild != NULL) {
        current = current->leftChild;
    }
    return current;
}
```

The `find_min()` method is called to find the minimum key in a subtree. You should convince yourself that the minimum value key in any binary search tree is the leftmost child of the tree. Therefore the `find_min()` method simply follows the `left_child` references in each node of the subtree until it reaches a node that does not have a left child.

```cpp
void spliceOut() {   // a method of the TreeNode class
    if (isLeaf()) {
        if (isLeftChild()) {
            parent->leftChild = NULL;
        } else {
            parent->rightChild = NULL;
        }
    } else if (hasAnyChild()) {
        TreeNode* child = (leftChild != NULL) ? leftChild : rightChild;
        if (isLeftChild()) {
            parent->leftChild = child;
        } else {
            parent->rightChild = child;
        }
        child->parent = parent;
    }
}
```

In [ ]:
load_anim("bst_delete")

We need to look at one last interface method for the binary search tree. Suppose that we would like to simply iterate over all the keys in the tree in order. You already know how to traverse a binary tree in order, using the inorder traversal algorithm. However, writing an iterator requires a bit more work since an iterator should return only one node each time the iterator is called.

`C++` has no `yield`, so the in-order iteration is written as a recursive traversal that visits the keys in sorted order:

```cpp
void inorder(TreeNode* node) {
    if (node != NULL) {
        inorder(node->leftChild);
        cout << node->value << " ";
        inorder(node->rightChild);
    }
}
```

`yield` is similar to `return` in that it returns a value to the caller. However, `yield` also takes the additional step of freezing the state of the function so that the next time the function is called it **continues** executing from the exact point it left off earlier. Functions that create objects that can be iterated are called <u>generator functions</u>.

At first glance you might think that the code is not recursive. However, remember that `__iter__` overrides the `for ... in` operation for iteration, so it really is recursive!

The complete class (including the three-case delete) is in `pythonds3/cppds/trees/binary_search_tree.cpp`. Here it is in action:

In [10]:
#include <iostream>
#include "dscpp/bst.hpp"   // TreeNode + BinarySearchTree (md listings above)
using namespace std;

int main() {
    BinarySearchTree myTree;
    myTree.put("a", "a");      myTree.put("q", "quick");
    myTree.put("b", "brown");  myTree.put("f", "fox");
    myTree.put("j", "jumps");  myTree.put("o", "over");
    myTree.put("t", "the");    myTree.put("l", "lazy");
    myTree.put("d", "dog");

    cout << myTree.get("q") << " " << myTree.get("l") << endl;
    cout << "There are " << myTree.length() << " items in this tree" << endl;
    myTree.remove("a");
    cout << "There are " << myTree.length() << " items in this tree" << endl;
    myTree.inorder(myTree.root);
    cout << endl;
    return 0;
}

quick lazy

There are 9 items in this tree

There are 8 items in this tree

brown dog fox jumps lazy over quick the 



https://visualgo.net/en/bst

#### Exercise 3: Using the `put()` and `in` method, write a sorting function that can sort a list in $O(n\log n)$ time in average case.

In [11]:
#include <iostream>
#include <vector>
#include "dscpp/bst.hpp"
using namespace std;

vector<string> treeSort(vector<string> values) {
    BinarySearchTree bst;
    vector<string> result;
    ____;   // 1. insert all elements (O(log n) each on average)
    ____;   // 2. in-order traversal visits keys in sorted order (O(n))
    return result;
}
int main() {
    for (string s : treeSort({"t", "a", "o", "j", "d", "b", "q", "f", "l"}))
        cout << s << " ";
    return 0;
}

/tmp/tmpxwhj3oyp.cpp: In function ‘std::vector<std::__cxx11::basic_string<char> > treeSort(std::vector<std::__cxx11::basic_string<char> >)’:
/tmp/tmpxwhj3oyp.cpp:11:5: error: ‘____’ was not declared in this scope
   11 |     ____;
      |     ^~~~



[C++ kernel] Error: Unable to compile the source code. Return error: 0x1.

Expected output: `a b d f j l o q t` — the in-order walk of a BST is always sorted.

## 8.14. Search Tree Analysis

Let's first look at the `put()` method. The limiting factor on its performance is the height of the binary tree. Recall from the vocabulary section that the height of a tree is the number of edges between the root and the deepest leaf node.

The height is the limiting factor because when we are searching for the appropriate place to insert a node into the tree, we will need to do at most one comparison at each level of the tree!

Note that if the keys are added in a random order, the height of the tree is going to be around $\log_2{n}$ where $n$ is the number of nodes in the tree. This is because if the keys are randomly distributed, about half of them will be less than the root and about half will be greater than the root.

Remember that in a binary tree there is one node at the root, two nodes in the next level, and four at the next. The number of nodes at any particular level is $2^d$ where $d$ is the depth of the level. The total number of nodes in a perfectly balanced binary tree is $2^{h+1}-1$, where $h$ represents the height of the tree.

A perfectly balanced tree has the same number of nodes in the left subtree as the right subtree. In a balanced binary tree, the worst-case performance of `put()` is $O(\log_2{n})$.

Notice that this is the inverse relationship to the calculation in the previous paragraph. So  $\log_2{n}$ gives us the height of the tree and represents the maximum number of comparisons that `put()` will need to do as it searches for the proper place to insert a new node.

Unfortunately it is possible to construct a search tree that has height $n$ simply by inserting the keys in sorted order! An example of this is shown below:

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/skewedTree.png" width="25%"/>
</center>

In this case the performance of the put method is $O(n)$.

Now that you understand that the performance of the `put()` method is limited by the height of the tree, you can probably guess that other methods, `get()`, `in`, and `del`, are limited as well. Since `get()` searches the tree to find the key, in the worst case the tree is searched all the way to the bottom and no key is found. 

At first glance `del` might seem more complicated since it may need to search for the successor before the deletion operation can complete. But remember that the worst-case scenario to find the successor is also just the height of the tree which means that you would simply double the work. Since doubling is a constant factor, it does not change worst-case analysis of $O(n)$ for an unbalanced tree.

In [ ]:
display_quiz(path+"bst.json", max_width=800)

<IPython.core.display.Javascript object>

## 8.15. Balanced Binary Search Trees 

In the previous section we looked at building a binary search tree. As we learned, the performance of the binary search tree can degrade to $O(n)$ for operations like `get()` and `put()` when the tree becomes unbalanced.

In this section we will look at a special kind of binary search tree that automatically makes sure that the tree remains balanced at all times. This tree is called an AVL tree.

An AVL tree implements the `Map` abstract data type just like a regular binary search tree; the only difference is in how the tree performs. To implement our AVL tree we need to keep track of a <u>balance factor</u> for each node in the tree.

We do this by looking at the heights of the left and right subtrees for each node. More formally, we define the balance factor for a node as the difference between the height of the left subtree and the height of the right subtree.

$$balance\_factor = height(left\_subtree) - height(right\_subtree)$$

Using the definition for balance factor given above, we say that a subtree is left-heavy if the balance factor is greater than zero. If the balance factor is less than zero, then the subtree is right-heavy. If the balance factor is zero, then the tree is perfectly in balance. 

For purposes of implementing an AVL tree and gaining the benefit of having a balanced tree, we will define a tree to be in balance if the balance factor is -1, 0, or 1. Once the balance factor of a node in a tree is outside this range we will need to have a procedure to bring the tree back into balance.

Below shows an example of an unbalanced right-heavy tree and the balance factors of each node:

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/unbalanced.png" width="20%"/>
</center>

## 8.16. AVL Tree Performance (Optional)

Before we proceed any further let's look at the result of enforcing this new balance factor requirement. Our claim is that by ensuring that a tree always has a balance factor of -1, 0, or 1 we can get better Big-O performance of key operations.

Let us start by thinking about how this balance condition changes the worst-case tree. There are two possibilities to consider, a left-heavy tree and a right-heavy tree. If we consider trees of heights 0, 1, 2, and 3, Figure below illustrates the most unbalanced left-heavy tree possible under the new rules.

<center>
    <img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/worstAVL.png" width="40%"/>
</center>

Looking at the total number of nodes in the tree we see that for a tree of height 0 there is 1 node, for a tree of height 1 there is $1 + 1 = 2$ nodes, for a tree of height 2 there are $1 + 1 + 2 = 4$, and for a tree of height 3 there are $1 + 2 + 4 = 7$. More generally the pattern we see for the number of nodes in a tree of height $N_h$ is:

$$N_h = 1 + N_{h-1} + N_{h-2}$$

This recurrence may look familiar to you because it is very similar to the Fibonacci sequence. We can use this fact to derive a formula for the height of an AVL tree given the number of nodes in the tree.

 Recall that for the Fibonacci sequence the $i^{th}$ Fibonacci number is given by:

$$\begin{split}F_0 & = 0 \\
F_1 & = 1 \\
F_i & = F_{i-1} + F_{i-2}  \text{ for all } i \ge 2\end{split}$$

An important mathematical result is that as the numbers of the Fibonacci sequence get larger and larger the ratio of $F_i / F_{i-1}$ becomes closer and closer to approximating the golden ratio $\Phi$ which is defined as $\Phi = \frac{1 + \sqrt{5}}{2}$.

We will simply use this equation to approximate $F_i$ as $F_i \approx \Phi^i/\sqrt{5}$. If we make use of this approximation we can rewrite the equation for $N_h$ as:

$$N_h = F_{h+3} - 1, h \ge 1$$

By replacing the Fibonacci reference with its golden ratio approximation we get:

$$N_h = \frac{\Phi^{h+2}}{\sqrt{5}} - 1$$

If we rearrange the terms, take the base 2 log of both sides, and then solve for $h$, we get the following derivation:

$$\begin{split}\log{(N_h+1)} &  = (h+2)\log{\Phi} - \frac{1}{2} \log{5} \\
h & = \frac{\log{(N_h+1)} - 2 \log{\Phi} + \frac{1}{2} \log{5}}{\log{\Phi}} \\
h &  \approx 1.44 \log{N_h}\end{split}$$ for large $N_h$

This derivation shows us that at any time the height of our AVL tree is equal to a constant (1.44) times the log of the number of nodes in the tree. This is great news for searching our AVL tree because it limits the search to $O(\log{n})$.

## 8.17. AVL Tree Implementation (Optional)

Let's look at how we will augment the procedure to insert a new key into the tree. Since all new keys are inserted into the tree as leaf nodes and we know that the balance factor for a new leaf is zero, there are no new requirements for the node that has just been inserted. 

But once the new leaf is added, we must update the balance factor of its parent. How this new leaf affects the parent's balance factor depends on whether the leaf node is a left child or a right child.

If the new node is a right child, the balance factor of the parent will be reduced by one. If the new node is a left child, then the balance factor of the parent will be increased by one. This rule can be applied recursively to the grandparent of the new node, and possibly to every ancestor, all the way up to the root of the tree!

Since this is a recursive procedure, let's examine the two base cases for updating balance factors:

- The recursive call has reached the root of the tree.

- The balance factor of the parent has been adjusted to zero. You should convince yourself that once a subtree has a balance factor of zero, then the balance of its ancestor nodes does not change.

We will implement the AVL tree as a subclass of `BinarySearchTree`. To begin, we will override the `_put()` method and write a new `update_balance()` helper method. 

```cpp
void _put(string key, string value, AVLTreeNode* currentNode) {
    if (key < currentNode->key) {
        if (currentNode->leftChild != NULL) {
            _put(key, value, currentNode->leftChild);
        } else {
            currentNode->leftChild = new AVLTreeNode(key, value, 0, currentNode);
            updateBalance(currentNode->leftChild);
        }
    } else {
        if (currentNode->rightChild != NULL) {
            _put(key, value, currentNode->rightChild);
        } else {
            currentNode->rightChild = new AVLTreeNode(key, value, 0, currentNode);
            updateBalance(currentNode->rightChild);
        }
    }
}
```

```cpp
void updateBalance(AVLTreeNode* node) {
    if (node->balanceFactor > 1 || node->balanceFactor < -1) {
        rebalance(node);
        return;
    }
    if (node->parent != NULL) {
        if (node->isLeftChild()) {
            node->parent->balanceFactor += 1;
        } else if (node->isRightChild()) {
            node->parent->balanceFactor -= 1;
        }
        if (node->parent->balanceFactor != 0) {
            updateBalance(node->parent);
        }
    }
}
```

This implements the recursive procedure we just described. It first checks to see if the current node is out of balance enough to require rebalancing.

If that is the case then the rebalancing is done and no further updating to parents is required. If the current node does not require rebalancing then the balance factor of the parent is adjusted. If the balance factor of the parent is nonzero then the algorithm continues to work its way up the tree toward the root by recursively calling `update_balance()` on the parent.

When a rebalancing of the tree is necessary, how do we do it? Efficient rebalancing is the key to making the AVL Tree work well without sacrificing performance. In order to bring an AVL Tree back into balance, we will perform one or more **rotations** on the tree.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/simpleunbalanced.png" width="40%" /></center>

Note the tree is out of balance with a balance factor of -2. To bring this tree into balance we will use a left rotation around the subtree rooted at node A.

To perform a left rotation we essentially do the following:

1. Promote the right child (B) to be the root of the subtree.

2. Move the old root (A) to be the left child of the new root.

3. If new root (B) already has a left child, then make it the right child of the new left child (A). Note: since the new root (B) was the right child of A, the right child of A is guaranteed to be empty at this point. This allows us to add a new node as the right child without any further consideration.

While this procedure is fairly easy in concept, the details of the code are a bit tricky since we need to move things around in just the right order so that all properties of a binary search tree are preserved. Furthermore, we need to make sure to update all of the parent pointers appropriately.

Let's look at a slightly more complicated tree to illustrate the right rotation where the left side of shows a tree that is left-heavy and with a balance factor of 2 at the root:

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/rightrotate1.png" width="40%" /></center>

To perform a right rotation we essentially do the following:

1. Promote the left child (C) to be the root of the subtree.

2. Move the old root (E) to be the right child of the new root.

3. If the new root (C) already has a right child (D) then make it the left child of the new right child (E). Note: since the new root (C) was the left child of E, the left child of E is guaranteed to be empty at this point. This allows us to add a new node as the left child without any further consideration.

```cpp
void rotateLeft(AVLTreeNode* rotationRoot) {
    AVLTreeNode* newRoot = rotationRoot->rightChild;
    rotationRoot->rightChild = newRoot->leftChild;
    if (newRoot->leftChild != NULL) {
        newRoot->leftChild->parent = rotationRoot;
    }
    newRoot->parent = rotationRoot->parent;
    if (rotationRoot->isRoot()) {
        root = newRoot;
    } else if (rotationRoot->isLeftChild()) {
        rotationRoot->parent->leftChild = newRoot;
    } else {
        rotationRoot->parent->rightChild = newRoot;
    }
    newRoot->leftChild = rotationRoot;
    rotationRoot->parent = newRoot;
    rotationRoot->balanceFactor = rotationRoot->balanceFactor + 1
                                  - min(newRoot->balanceFactor, 0);
    newRoot->balanceFactor = newRoot->balanceFactor + 1
                             + max(rotationRoot->balanceFactor, 0);
}
```

The code is for the left rotation (the `rotate_right()` method is symmetrical).

We create a temporary variable to keep track of the new root of the subtree. As we said before, the new root is the right child of the previous root. Now that a reference to the right child has been stored in this temporary variable, we replace the right child of the old root with the left child of the new.

The next step is to adjust the parent pointers of the two nodes. If `new_root` has a left child then the new parent of the left child becomes the old root. The parent of the new root is set to the parent of the old root. If the old root was the root of the entire tree then we must set the root of the tree to point to this new root. 

Otherwise, if the old root is a left child then we change the parent of the left child to point to the new root; otherwise we change the parent of the right child to point to the new root. Finally we set the parent of the old root to be the new root. 

Finally, the bottome parts require some explanation. In these lines we update the balance factors of the old and the new root. Since all the other moves involve moving entire subtrees, the balance factors of all other nodes are unaffected by the rotation. 

But how can we update the balance factors without completely recalculating the heights of the new subtrees? Figure below and the following derivation should convince you that these lines are correct.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/bfderive.png" width="40%" /></center>

Figure above shows a left rotation. B and D are the pivotal nodes and A, C, E are their subtrees. Let $h_x$ denote the height of a particular subtree rooted at node $x$. By definition we know the following:

$$\begin{split}new\_bal(B) = h_A - h_C \\
old\_bal(B) = h_A - h_D\end{split}$$

But we know that the old height of D can also be given by $1 + max(h_C, h_E)$, that is, the height of D is one more than the maximum height of its two children. Remember that $h_C$ and $h_E$ have not changed. So, let us substitute that in to the second equation, which gives us

$$old\_bal(B) = h_A - (1 + max(h_C,h_E))$$

and then subtract the two equations. The following steps do the subtraction and use some algebra to simplify the equation for $new\_bal(B)$.

$$\begin{split}new\_bal(B) - old\_bal(B) = h_A - h_C - (h_A - (1 + max(h_C,h_E))) \\
new\_bal(B) - old\_bal(B) = h_A - h_C - h_A + (1 + max(h_C,h_E)) \\
new\_bal(B) - old\_bal(B) = h_A  - h_A + 1 + max(h_C,h_E) - h_C  \\
new\_bal(B) - old\_bal(B) =  1 + max(h_C,h_E) - h_C\end{split}$$

Next we will move $old\_bal(B)$ to the right-hand side of the equation and make use of the fact that $max(a,b)-c = max(a-c, b-c)$:

$$\begin{split}new\_bal(B) = old\_bal(B) + 1 + max(h_C - h_C ,h_E - h_C) \\\end{split}$$

But $h_E - h_C$ is the same as $-old\_bal(D)$. So we can use another identity that says $max(-a,-b) = -min(a,b)$. So we can finish our derivation of $new\_bal(B)$ with the following steps:

$$\begin{split}new\_bal(B) = old\_bal(B) + 1 + max(0 , -old\_bal(D)) \\
new\_bal(B) = old\_bal(B) + 1 - min(0 , old\_bal(D)) \\\end{split}$$

Now we have all of the parts in terms that we readily know. If we remember that B is `rotationRoot` and D is `newRoot` then we can see this corresponds exactly to the statement

```cpp
rotationRoot->balanceFactor = rotationRoot->balanceFactor + 1
                              - min(newRoot->balanceFactor, 0);
```

A similar derivation gives us the equation for the updated node D as well as the balance factors after a right rotation. We leave these as an exercise for you.

Now you might think that we are done. We know how to do our left and right rotations, and we know when we should do a left or right rotation. 

But take a look at Figure below. Since node A has a balance factor of -2 we should do a left rotation. But what happens when we do the left rotation around A?

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/hardunbalanced.png" width="12%" /></center>

Below shows us that after the left rotation we are now out of balance the other way. If we do a right rotation to correct the situation we are right back where we started!

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/badrotate.png" width="12%" /></center>

To correct this problem we must use the following set of rules:

1. If a subtree needs a left rotation to bring it into balance, first check the balance factor of the right child. If the right child is left-heavy, then do a right rotation on right child followed by the original left rotation.

2. If a subtree needs a right rotation to bring it into balance, first check the balance factor of the left child. If the left child is right-heavy, then do a left rotation on the left child followed by the original right rotation.

Figure below shows how these rules solve the dilemma we encountered. Starting with a right rotation around node C puts the tree in a position where the left rotation around A brings the entire subtree back into balance.

<center><img src="https://raw.githubusercontent.com/phonchi/nsysu-math208/refs/heads/main/static_files/presentations/imgs/rotatelr.png" width="40%" /></center>

The code that implements these rules can be found in our rebalance method:

```cpp
void rebalance(AVLTreeNode* node) {
    if (node->balanceFactor < 0) {
        if (node->rightChild->balanceFactor > 0) {
            rotateRight(node->rightChild);
            rotateLeft(node);
        } else {
            rotateLeft(node);
        }
    } else if (node->balanceFactor > 0) {
        if (node->leftChild->balanceFactor < 0) {
            rotateLeft(node->leftChild);
            rotateRight(node);
        } else {
            rotateRight(node);
        }
    }
}
```

By keeping the tree in balance at all times, we can ensure that the get method will run in order $O(log_2(n))$ time. But the question is at what cost to our put method? Let us break this down into the operations performed by `put()`. 

Since a new node is inserted as a leaf, updating the balance factors of all the parents will require a maximum of $log_2(n)$ operations, one for each level of the tree. If a subtree is found to be out of balance, a maximum of two rotations are required to bring the tree back into balance. But each of the rotations works in $O(1)$ time, so even our put operation remains $O(log_2(n))$.

At this point we have implemented a functional AVL tree, unless you need the ability to delete a node. We leave the deletion of the node and subsequent updating and rebalancing as an exercise for you.

In [ ]:
display_quiz(path+"avl.json", max_width=800)

<IPython.core.display.Javascript object>

A complete C++ AVL tree combines the BST above with the balance-factor bookkeeping and rotations shown here — see the cppds textbook's AVL chapter for the full listing, and try inserting 1..5 to watch the left rotations keep the tree height logarithmic.

## 8.18. Summary of Map ADT Implementations

Over the past two chapters we have looked at several data structures that can be used to implement the map abstract data type: a binary search on a list, a hash table, a binary search tree, and a balanced binary search tree. To conclude this section, let’s summarize the performance of each data structure for the key operations defined by the map ADT:

<div align="center">

| operation |   Sorted List  | Hash Table | Binary Search Tree |    AVL Tree    |
|:---------:|:--------------:|:----------:|:------------------:|:--------------:|
|    `put()`    |     $O(n)$     |   $O(1)$   |       $O(n)$       | $O(\log_2{n})$|
|    `get()`    | $O(\log_2{n})$ |   $O(1)$   |       $O(n)$       | $O(\log_2{n})$|
|     `in()`    | $O(\log_2{n})$ |   $O(1)$   |       $O(n)$       | $O(\log_2{n})$|
|    `del()`    |     $O(1)$     |   $O(1)$   |       $O(n)$       | $O(\log_2{n})$|

</div>

In [ ]:
from jupytercards import display_flashcards
fpath = "flashcards/"
display_flashcards(fpath + 'ch9.json')

<IPython.core.display.Javascript object>

## References

1. Textbook: *Problem Solving with Algorithms and Data Structures using C++* (cppds), Chapter 8 (Trees and Tree Algorithms) — https://runestone.academy/ns/books/published/cppds/index.html

## Key terms

Here's a brief definition for each of the key terms:

- **Trees**: A hierarchical data structure consisting of nodes, where each node contains data and references to child nodes, facilitating efficient data organization and retrieval.

- **Node**: A fundamental part of a tree that stores data and contains references to other nodes (children). Each node can represent a data point or a position in the structure.

- **Edge**: The connection between two nodes in a tree, representing the relationship or link from a parent node to a child node.

- **Root**: The topmost node in a tree, from which all other nodes descend. It does not have a parent node.

- **Path**: A sequence of nodes and edges connecting a node with a descendant, showing the route from one node to another within the tree.

- **Children**: Nodes that are directly connected and descend from another node (the parent) within a tree.

- **Parent**: A node that has one or more nodes (children) directly under it in the hierarchy.

- **Sibling**: Nodes that share the same parent and reside at the same level within a tree.

- **Subtree**: A portion of a tree structure consisting of a node and all its descendants, forming a tree itself.

- **Leaf node**: A node with no children, representing an endpoint of a tree.

- **Level**: The layer or depth of a node, starting at zero for the root and increasing by one for each step down.

- **Height**: The total number of levels in a tree, or the length of the longest path from the root to a leaf node.

- **Binary Tree**: A type of tree where each node has at most two children, typically called the left and right children.

- **List of Lists**: A method for implementing a tree using a nested list structure where each list represents a node and its children.

- **Nodes and References**: A tree implementation method where each node contains its data and explicit links to its child nodes.

- **Parse Tree**: A tree representing the syntactic structure of a string according to a formal grammar, used in compilers.

- **Expression Tree**: A binary tree used to represent expressions where internal nodes are operators and leaf nodes are operands.

- **Preorder**: A tree traversal where each node is processed before its child nodes (root, left, right).

- **Postorder**: A tree traversal where each node is processed after its child nodes (left, right, root).

- **Inorder**: A traversal for binary trees where nodes are processed in the order: left-child, root, right-child.

- **Priority Queue**: An abstract data type where each element has a priority, and elements are served based on that priority.

- **Binary Heap**: A binary tree-based structure that satisfies the heap property, where nodes are ordered relative to their children.

- **Complete Binary Tree**: A binary tree where every level except possibly the last is filled, and all nodes are as far left as possible.

- **Heap Structure Property**: A requirement that a heap must be a complete binary tree to ensure balance.

- **Heap Order Property**: A requirement maintaining a specific order between a node and its children (max-heap or min-heap).

- **Max-Heap**: A heap where the value of each node is greater than or equal to its children, keeping the maximum value at the root.

- **Min-Heap**: A heap where the value of each node is less than or equal to its children, keeping the minimum value at the root.

- **Binary Search Tree (BST)**: A binary tree where each node's key is greater than all keys in its left subtree and less than those in its right subtree.

- **BST Property**: The defining rule for a BST: left subtree elements are less than the node, and right subtree elements are greater.

- **Successor**: The node that immediately follows a given node in an inorder traversal of a binary search tree.

- **Generator Function**: A function that returns an iterator and yields values one at a time, pausing between each execution.

- **Balanced Binary Search Tree**: A BST that automatically adjusts to maintain a low height, ensuring minimal path lengths.

- **Balance Factor**: In AVL trees, the difference between the heights of a node's left and right subtrees.

- **AVL Tree**: A self-balancing BST where the height of subtrees differs by at most one, ensuring logarithmic time complexity.
